# TP N°4 : Régression linéaire simple et multiple

**Département :** IA & SD
**Section :** M2 Bioinformatique
**Module :** FD2
**Réalisé par :** BERBICHE AYMEN / SLIFI BAYA INES
**Date de remise :** 27/11/2026

## 1. Importation des bibliothèques

- `numpy` et `pandas` pour le calcul et la manipulation des données.
- `matplotlib` et `seaborn` pour les graphiques.
- `load_diabetes` pour charger le jeu de données.
- `train_test_split` pour séparer les données en apprentissage et test.
- `LinearRegression` pour construire les modèles de régression.
- `mean_absolute_error`, `mean_squared_error` et `r2_score` pour évaluer les modèles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Chargement et premières lignes

Le jeu de données est chargé avec `load_diabetes(as_frame=True)`, ce qui renvoie un tableau pandas. La variable `X` contient les 10 variables explicatives et `y` la progression de la maladie (`target`).

Le jeu comporte 442 patients et 10 variables explicatives. Les variables sont déjà standardisées par Scikit-learn : leur moyenne est proche de 0 et leur écart-type d'environ 0,048.

In [ ]:
data = load_diabetes(as_frame=True)
df = data.frame
X = data.data
y = data.target

print(f"Dimensions : {df.shape[0]} patients, {X.shape[1]} variables explicatives")
print(f"Variables explicatives : {list(X.columns)}")
df.head()

## 3. Statistiques descriptives, valeurs manquantes et doublons

Le tableau présente la moyenne, la médiane (colonne `50%` de `describe`), l'écart-type, le minimum et le maximum de chaque variable. Les variables explicatives ont une moyenne nulle et un écart-type d'environ 0,048, car elles sont standardisées. La cible `target` varie de 25 à 346, avec une moyenne de 152,1 et une médiane de 140,5.

Le jeu de données ne contient ni valeurs manquantes ni doublons, aucun nettoyage n'est donc nécessaire.

In [ ]:
stats = df.describe().T[['mean', '50%', 'std', 'min', 'max']]
stats.columns = ['moyenne', 'médiane', 'écart-type', 'min', 'max']
print(stats.round(3))

print("Valeurs manquantes :", df.isna().sum().sum())
print("Doublons :", df.duplicated().sum())

: 

## 4. Analyse des corrélations

La matrice de corrélation mesure le lien linéaire entre chaque paire de variables, entre −1 et 1.

La cible est surtout liée au BMI (r = 0,59), à `s5` (r = 0,57), à `bp` (r = 0,44) et à `s4` (r = 0,43). La variable `sex` est presque sans lien avec la progression (r = 0,04).

Certaines variables explicatives sont fortement liées entre elles, notamment `s1` et `s2` (r = 0,90) ainsi que `s3` et `s4` (r = −0,74). Cette multicolinéarité rendra les coefficients du modèle multiple plus difficiles à interpréter.

In [ ]:
print("Corrélation avec la cible :")
print(df.corr()['target'].sort_values(ascending=False).round(3))

plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

: 

## 5. Relations entre les variables et la cible

Chaque graphique présente une variable explicative en abscisse et la progression (`target`) en ordonnée.

Les variables `bmi`, `s5` et `bp` montrent une tendance croissante : quand elles augmentent, la progression tend à augmenter. Les autres variables présentent des nuages plus dispersés. `sex` ne prend que deux valeurs, d'où les deux colonnes de points.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, col in zip(axes.flat, X.columns):
    ax.scatter(df[col], y, alpha=0.5, s=12)
    ax.set_xlabel(col)
    ax.set_ylabel('target')
    ax.set_title(f"{col} vs target")
plt.tight_layout()
plt.show()

## 6. Séparation apprentissage / test

Les données sont séparées en deux ensembles : 80 % des patients servent à entraîner le modèle, 20 % à le tester. Le paramètre `random_state=42` fixe le tirage, ce qui rend les résultats reproductibles.

On obtient 353 patients pour l'apprentissage et 89 pour le test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Apprentissage : {X_train.shape[0]} lignes")
print(f"Test          : {X_test.shape[0]} lignes")

## 7. Régression linéaire simple (BMI → target)

Le modèle cherche la droite `Y = aX + b` qui s'approche le mieux des points d'apprentissage, où X est le BMI, `a` la pente et `b` la constante.

On obtient **Y = 998,58 × BMI + 152,00**. La pente est positive : quand le BMI augmente, la progression augmente aussi. Comme le BMI est standardisé, la pente se lit dans cette unité et non en kg/m².

Le graphique montre une tendance générale nette, mais une forte dispersion autour de la droite, ce qui explique le R² faible obtenu plus loin (0,233).

In [ ]:
X_train_s = X_train[['bmi']]
X_test_s = X_test[['bmi']]

model_simple = LinearRegression()
model_simple.fit(X_train_s, y_train)

a = model_simple.coef_[0]
b = model_simple.intercept_
print(f"Équation : Y = {a:.2f} * BMI + {b:.2f}")

y_pred_simple = model_simple.predict(X_test_s)

xs = pd.DataFrame(np.linspace(X['bmi'].min(), X['bmi'].max(), 100), columns=['bmi'])

plt.figure(figsize=(8, 5))
plt.scatter(X_test_s, y_test, color='steelblue', alpha=0.6, label='Données de test')
plt.plot(xs['bmi'], model_simple.predict(xs), color='red', linewidth=2,
         label='Droite de régression')
plt.xlabel('BMI (standardisé)')
plt.ylabel('Progression (target)')
plt.title('Régression linéaire simple')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Évaluation des performances

Le sujet demande Accuracy, Precision, Recall, F1-score et la matrice de confusion. Ces mesures sont destinées à la classification. La cible `target` étant continue, on utilise les mesures de régression : MAE (erreur absolue moyenne), MSE et RMSE (erreur quadratique), et R² (part de la variance expliquée), ajusté au nombre de variables.

| Modèle | MAE | MSE | RMSE | R² | R² ajusté |
|---|---|---|---|---|---|
| Simple (BMI) | 52,26 | 4 061,83 | 63,73 | 0,233 | 0,225 |
| Multiple (10 variables) | 42,79 | 2 900,19 | 53,85 | 0,453 | 0,382 |

Le modèle multiple est meilleur sur les quatre mesures. Il réduit le RMSE de 63,7 à 53,9 et le MAE de 52,3 à 42,8. Le R² passe de 0,233 à 0,453 : le BMI seul explique environ 23 % de la variance de la progression, les dix variables environ 45 %. Le R² ajusté (0,225 à 0,382) confirme que le gain ne vient pas seulement du nombre de variables.

In [ ]:
def evaluer(y_true, y_pred, n_features, nom):
    n = len(y_true)
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    return {
        'Modèle': nom,
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R²': r2,
        'R² ajusté': 1 - (1 - r2) * (n - 1) / (n - n_features - 1),
    }

resultats = pd.DataFrame([
    evaluer(y_test, y_pred_simple, 1, 'Simple (BMI)'),
    evaluer(y_test, y_pred_multi, X.shape[1], 'Multiple (10 variables)'),
])
print(resultats.round(3).to_string(index=False))

## 10. Valeurs réelles et valeurs prédites

Chaque point représente un patient du jeu de test. Plus il est proche de la ligne rouge en pointillés, meilleure est la prédiction.

Dans les deux cas, les progressions élevées sont sous-estimées et les faibles sont surestimées : les modèles ont tendance à prédire des valeurs proches de la moyenne. Le modèle multiple est plus proche de la prédiction parfaite, mais une part importante de la variance reste non expliquée.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, pred, titre in zip(
    axes,
    [y_pred_simple, y_pred_multi],
    ['Simple (BMI)', 'Multiple (10 variables)']
):
    ax.scatter(y_test, pred, alpha=0.6, color='steelblue')
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, 'r--', label='Prédiction parfaite')
    ax.set_xlabel('Valeurs réelles')
    ax.set_ylabel('Valeurs prédites')
    ax.set_title(titre)
    ax.legend()
plt.tight_layout()
plt.show()

## 11. Conclusion

Le BMI seul explique environ 23 % de la variance de la progression de la maladie. Les dix variables explicatives en expliquent environ 45 %, ce qui rend la régression multiple nettement meilleure.

Deux limites doivent être signalées. D'abord, la forte corrélation entre certaines variables (`s1` et `s2`, `s3` et `s4`) rend les coefficients difficiles à interpréter individuellement. Ensuite, le modèle est linéaire : il ne capte pas d'éventuelles relations non linéaires.

Des pistes d'amélioration seraient une validation croisée (`cross_val_score`) pour vérifier la stabilité des résultats, une régularisation (Ridge ou Lasso) pour stabiliser les coefficients, et l'utilisation de modèles non linéaires.